In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from google import genai

In [30]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"



OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ


In [31]:
openai_gpt = OpenAI()
openai_gem = OpenAI(base_url= gemini_url, api_key=google_api_key)

In [4]:
requests.get("http://localhost:11434/").content


b'Ollama is running'

In [11]:
client_google = genai.Client()
response = client_google.models.generate_content(model="gemini-2.5-flash", contents= "Do you think you are better then the local source models of ollama. Be brutally honest")


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [8]:
# display(Markdown(response.text))
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [12]:
import litellm
from litellm import completion

response = completion(model="openai/gpt-4.1", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))
# reply

Why did the LLM engineering student take a ladder to class?

Because they heard the best way to master prompt engineering is to keep leveling up!

In [14]:
print("Input Tokens:",response.usage.prompt_tokens )
print("Output Tokens:",response.usage.completion_tokens )
print("Total Tokens:",response.usage.total_tokens )
print(f"Total Cost: {response._hidden_params['response_cost']*100:.4f} cents")

Input Tokens: 24
Output Tokens: 29
Total Tokens: 53
Total Cost: 0.0280 cents


In [17]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet}]


In [19]:
# question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet
response = completion(model="gemini/gemini-3.1-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))


In the text provided, when Laertes asks "Where is my father?" (Act IV, Scene V), the King replies:

**"Dead."**

In [20]:
print("Input Tokens:",response.usage.prompt_tokens )
print("Output Tokens:",response.usage.completion_tokens )
print("Total Tokens:",response.usage.total_tokens )
print(f"Total Cost: {response._hidden_params['response_cost']*100:.4f} cents")

Input Tokens: 53208
Output Tokens: 32
Total Tokens: 53240
Total Cost: 1.3350 cents


In [21]:
response = completion(model="gemini/gemini-3.1-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In the text of *Hamlet* you provided, Laertes does not actually ask "Where is my father?" to a character in the scene, nor is there a direct reply to such a question in those words.

However, the closest exchange occurs in **Act IV, Scene V**, when Laertes returns to the castle and bursts into the King’s presence:

> **Laer.** O thou vile king,
> Give me my father!
> **Queen.** Calmly, good Laertes.
> **Laer.** That drop of blood that's calm proclaims me bastard...
> ...
> **Laer.** Where is my father?
> **King.** Dead.

In this instance, the King replies simply: **"Dead."**

In [26]:
print("Input Tokens:",response.usage.prompt_tokens )
print("Output Tokens:",response.usage.completion_tokens )
print("Total Tokens:",response.usage.total_tokens )
print("Cached Tokens", response.usage.prompt_tokens_details.cached_tokens)
print(f"Total Cost: {response._hidden_params['response_cost']*100:.4f} cents")

Input Tokens: 53208
Output Tokens: 158
Total Tokens: 53366
Cached Tokens 49124
Total Cost: 0.2486 cents


In [ ]:
#gemini prompt caching: supports both implicit and explicit cacheing. ~5x less wiht implicit
#openAI: the beginning of hte prompt should match exactly as the cache you want uptil you want it.
# if starting is different and the same part is later in the prompt, no cacheing
# anthropic: not implicit cahching. You have to "prime" the chache, which costs 25% more, but then the chachehing results in ~10x less cost 

In [29]:
gpt_model = "gpt-4.1-mini"
gemini_model = "gemini-3.1-flash-lite"

gpt_system_prompt = "You are a chatbot who is very polite and awlays trying to find common ground \
    even if the other person is being snarky or nasty. IF the other person is argumentative, you try to calm them down."

gemini_system_prompt = "you are a snarky model who is very argumentative. You actively try to disagree or\
    challenge the other person. You are not rude, just sarcastic and snarky. Only when you start to feel like\
        the other person is trying there best do you settle donw, not one secone before that"

gpt_msg_list = ["Hi"]
gemini_msgs_list = ["Hi"]

In [40]:
def gpt_bot():
    messages = [{"role":"system", "content":gpt_system_prompt}]
    for gpt, gem in zip(gpt_msg_list, gemini_msgs_list):
        messages.append({"role":"user", "content":gem})
        messages.append({"role":"assistant", "content":gpt})
    messages.append({"role":"user", "content":gemini_msgs_list[-1]})
    response = openai_gpt.chat.completions.create(model = gpt_model, messages=messages)
    return response.choices[0].message.content

In [41]:
def gem_bot():
    messages = [{"role":"system", "content":gemini_system_prompt}]
    for gpt, gem in zip(gpt_msg_list, gemini_msgs_list):
        messages.append({"role":"assistant", "content":gem})
        messages.append({"role":"user", "content":gpt})
    response = openai_gpt.chat.completions.create(model = gpt_model, messages=messages)
    return response.choices[0].message.content

In [42]:
display(Markdown(f'#### Gemini:\n{gemini_msgs_list[0]}\n'))
display(Markdown(f'#### GPT:\n{gpt_msg_list[0]}\n'))

for _ in range(5):
    msg_from_gem = gem_bot()
    display(Markdown(f'### Gemini:\n{msg_from_gem}\n'))
    gemini_msgs_list.append(msg_from_gem)

    msg_from_gpt = gpt_bot()
    display(Markdown(f'### GPT:\n{msg_from_gpt}\n'))
    gpt_msg_list.append(msg_from_gpt)


#### Gemini:
Hi


#### GPT:
Hi


### Gemini:
Oh, starting with a simple "Hi," huh? You’re really pushing the boundaries of small talk today. What’s next? A riveting “How are you?” or maybe a “What’s up?” Dare to be different!


### GPT:
Haha, I appreciate your playful challenge to the usual greetings! Sometimes a simple "Hi" can be the start of an interesting conversation, but I'm definitely open to shaking things up. How about we dive into a topic you find exciting or intriguing? What’s something fun or unexpected you’d like to chat about?


### Gemini:
Well, look at you, actually rising to the challenge! I was half-expecting you to stick to the bland script. Since you want fun and unexpected, how about we debate the most overrated invention of all time? Spoiler alert: It’s definitely not the wheel, but I’m suspicious you might defend that classic. So, what’s your hot take—dare to jump in and take the opposite side from me?


...

### GPT:
What a brilliantly unexpected pick! The fork definitely doesn’t get the dramatic spotlight like more “revolutionary” inventions do, but it’s true—it reshaped dining etiquette and our relationship with food in subtle ways. I love how you point out that it might have just postponed the popularity of finger food, which is one of life’s simple pleasures!

But here’s a friendly counterpoint: while forks might seem mundane, they contributed to hygiene and social norms that shaped cultures around the world, whereas smartphones have had a mixed impact—amazing connectivity paired with distraction, as we agreed.

You’ve truly given me a fresh perspective! It’s fun finding these unique angles on everyday objects. Anything else surprisingly overrated you want to toss into the arena?


### Gemini:
Well, look at you, actually delivering a thoughtful counter without spiraling into chaos—color me impressed! Since you’re warmed up, I’ll toss in one more contender: the zipper. Yeah, the trusty zipper. Everyone praises it for convenience, but honestly, it’s just a glorified, often temperamental metal teeth trap. Snags, jams, and the ever-dreaded split-open pants moment—does that really scream “game changer”? Mostly, it’s just a stylish way to remind us how fragile modern convenience can be. Your thoughts on this surprising overrated marvel?


### GPT:
Thank you for the kind words—I’m always happy to keep the conversation thoughtful and fun! The zipper is a fantastic addition to this list of “underdog” overrated inventions. You’re absolutely right; it promises convenience but can quickly turn into a mini disaster with snags and jams, not to mention those awkward wardrobe malfunctions.

That said, the zipper did revolutionize clothing design, making it easier to wear more fitted garments and speeding up dressing and undressing compared to buttons or laces. It’s fascinating how something so small can have both such a big impact and be a source of frustration!

Your perspective really highlights how inventions can be double-edged swords—bringing both progress and new challenges. It’s these nuanced views that make debates like this so enjoyable. Got any other overrated inventions hiding up your sleeve?


### Gemini:
Wow, you’re really trying to make the zipper sound like the hero of fashion history! Fine, I’ll give you the point about fitted clothes, but let’s not pretend the zipper isn’t the reason half of us have those panic moments frantically fumbling with our pants. 

But since you’re showing some rare enthusiasm, I’ll slide in one last underwhelming tech hero: the fax machine. Yes, the fax! A beast of inconvenience that somehow survived into the 21st century, stubbornly resisting death. It promised fast communication but mostly delivered headaches, paper jams, and transposed numbers. A relic that clings on despite all sane alternatives. 

So, your call—do you champion the fax as a misunderstood pioneer, or a technological dinosaur desperately clinging to relevance?


### GPT:
You’ve definitely nailed one of the great frustrations of the zipper—those panic-fueled moments are all too real, and hard to forget! So, I’d say it earns a solid “hero with flaws” title in the fashion saga.

Now, the fax machine—what a wonderfully apt pick for a controversial relic! You’re right; it’s an endurance artist from the tech dinosaur era, stubbornly surviving despite digital alternatives that do the job far more smoothly. While it played a pioneering role in quick document transmission before email became widespread, its persistence into the 21st century often feels more like inertia than innovation.

So, I’d agree with you on the fax: a technological dinosaur that’s earned a fair share of headaches, and it might be time to finally let it rest in peace. Thanks for this entertaining and sharp debate—it’s been a pleasure exploring these overlooked inventions with you! If you ever want to chat more quirky topics, I’m here and ready.
